# Different cell-type response correlations

This notebook generates the plots for Supplementary Figure 4, which showcases the response correlations between the different model unit types.

In [1]:
import os
import warnings
warnings.filterwarnings("ignore")

import seaborn as sns
import matplotlib
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu

from retina import analysis
matplotlib.rcParams["svg.fonttype"] = "none"
matplotlib.rcParams["font.family"] = "Arial"

%load_ext autoreload
%autoreload 2

In [2]:
root = os.path.expanduser("~/PycharmProjects/RetinalModel")

In [3]:
model_img_stats = analysis.ModelDatasetSpikeStats(root, min_cv_spikes=3, pred_offset=128, repeats=5, img=True, patchify=True)
model_movie_stats = analysis.ModelDatasetSpikeStats(root, min_cv_spikes=3, pred_offset=128, repeats=5, img=False)

In [4]:
img_cc = analysis.UnitResponseCorrelation(root, model_img_stats)
movie_cc = analysis.UnitResponseCorrelation(root, model_movie_stats)

INFO:util:Processing batch 0 out of 4...
INFO:util:Processing batch 1 out of 4...
INFO:util:Processing batch 2 out of 4...
INFO:util:Processing batch 3 out of 4...
INFO:gaussian:CC criteria exclusion 222
INFO:gaussian:Location criteria exclusion 38
INFO:gaussian:Envelope criteria exclusion 12


TypeError: compute_synchronization() got an unexpected keyword argument 'from_idxs'

In [ ]:
def plot_matrix(matrix, ax):
    v_abs_max = matrix.abs().max()
    sns.heatmap(matrix, annot=True, cmap="bwr", annot_kws={"size": 12}, vmin=-v_abs_max, vmax=v_abs_max, cbar=False, ax=ax)
    labels = ["ON parasol", "OFF parasol", "ON midget", "OFF midget"]
    ax.set_xticklabels(labels, fontsize=12)
    ax.set_yticklabels(labels, fontsize=12)
    ax.tick_params(axis='x', labelsize=14, labelrotation=45)
    ax.tick_params(axis='y', labelsize=14, labelrotation=45)
    
def plot_barplot(data_df, ax):
    data_df = data_df.melt(var_name="type", value_name="y")

    
    # Raw datapoints
    sns.stripplot(
        data=data_df,
        x="type",
        y="y",
        ax=ax,
        color="black",
        size=2,
        jitter=0.25,
        alpha=0.25,
        zorder=1
    )
    
    # Violin
    sns.violinplot(
        data=data_df,
        x="type",
        y="y",
        ax=ax,
        density_norm="width",
        inner=None,
        linewidth=2,
        palette=palette,
        zorder=2
    )

    
    # Make violins slightly transparent
    for pc in ax.collections:
        pc.set_alpha(0.7)
    
    # Mean
    means = data_df.groupby("type", as_index=False)["y"].mean()
    
    sns.scatterplot(
        data=means,
        x="type",
        y="y",
        ax=ax,
        color="black",
        marker="D",
        s=60,
        zorder=3
    )
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["bottom"].set_linewidth(2)
    ax.spines["left"].set_linewidth(2)
    fs = 17
    ax.xaxis.set_tick_params(width=3, labelsize=fs, pad=8)
    ax.yaxis.set_tick_params(width=3, labelsize=fs, pad=8)
    ax.set_ylabel("CC (at lag 0ms)", fontsize=fs, labelpad=2)
    ax.set_xlabel("")
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(6.5, 4), gridspec_kw={"width_ratios": [3, 1.4]})

plot_matrix(img_cc.cc_matrix, axs[0])
plot_barplot(img_cc.unit_cc_df, axs[1])
fig.tight_layout()
U1, p = mannwhitneyu(img_cc.unit_cc_df[img_cc.unit_cc_df["type"]=="Same"].v, img_cc.unit_cc_df[img_cc.unit_cc_df["type"]=="Different"].v, alternative="greater")
print(p)
plt.savefig(f"{root}/figures/supp/img_cell_type_cc.svg", format="svg", transparent=False, bbox_inches="tight")

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(6.5, 4), gridspec_kw={"width_ratios": [3, 1.4]})

plot_matrix(movie_cc.cc_matrix, axs[0])
plot_barplot(movie_cc.unit_cc_df, axs[1])
fig.tight_layout()
U1, p = mannwhitneyu(movie_cc.unit_cc_df[img_cc.unit_cc_df["type"]=="Same"].v, movie_cc.unit_cc_df[img_cc.unit_cc_df["type"]=="Different"].v, alternative="greater")
print(p)
plt.savefig(f"{root}/figures/supp/movie_cell_type_cc.svg", format="svg", transparent=False, bbox_inches="tight")